# Model selection

$m_{\text{stop}}$ is the primary hyperparameter. This chapter
covers how it is chosen and the resampling machinery behind `cvrisk`.

## Early stopping as regularization

The boosting path traces a sequence of models of increasing complexity. Under
smooth convex loss and a sufficiently small step, training risk decreases.
Expected test risk often falls and later rises, but a finite-sample curve need
not be smooth, U-shaped, or turn within the budget. Resampling estimates held-out risk along the path, and the selected
$m_{\text{stop}}$ minimizes that estimate.

```{note}
A descent direction guarantees a decrease only for a sufficiently small step;
nonsmooth and calibrated families require checking the recorded path. A flat
or boundary cross-validation curve can reflect limited resolution, fold
variation, a useful plateau, or an insufficient path budget.
See the troubleshooting section below.
```

## Cross-validated risk

`cvrisk` estimates the out-of-sample risk **as a function of $m$** in one pass:

1. Draw $B$ resampling folds, each defining in-bag weights $w^{(b)}$ (out-of-bag
   points get weight 0).
2. For each fold $b$, fit the full path to $m_{\max}$ on the in-bag weights.
   Penalized learners **recompute** their $\lambda$ from `df` under
   $w^{(b)}$ (see [](penalties_df.ipynb)); this recomputation keeps the
   target `df` constant across folds even though the effective sample size
   changes.
3. At every $m$, evaluate the risk on the out-of-bag points:
   $r_b(m) = \sum_i v_i^{(b)}\rho(y_i, \hat f^{[m]}_b) / \sum_i v_i^{(b)}$,
   where $v^{(b)}$ explicitly marks and weights evaluation rows. Bootstrap
   multiplicities are training weights; subtracting them from base weights
   can be negative and is not an out-of-bag definition.
4. The cross-validated risk is the fold mean $\bar r(m) = \frac1B \sum_b r_b(m)$;
   the chosen iteration is $\hat m = \argmin_m \bar r(m)$.

```{list-table} Resampling schemes
:header-rows: 1

* - `type`
  - Folds
  - Use case
* - `"bootstrap"`
  - $B$ bootstrap samples (in-bag = multiplicity weights)
  - Default; stable estimate, slightly optimistic
* - `"kfold"`
  - $B$-fold cross-validation
  - Conservative; each observation is out-of-bag exactly once
* - `"subsampling"`
  - $B$ random splits at a fixed in-bag fraction
  - Flexible; controls the in-bag fraction
```

### Fold membership across the schemes

The three resampling schemes differ in how they assign observations to in-bag
and out-of-bag sets:

- **Bootstrap**: each observation is independently drawn into the in-bag
  set with replacement. On average, 63.2% of observations appear at least
  once in a given fold (the "bootstrap inclusion probability"
  $1 - (1 - 1/n)^n \to 1 - e^{-1}$). The remaining 36.8% are out-of-bag.
  Observations can appear multiple times — the weight is their
  multiplicity.

- **K-fold**: the data is split into $K$ disjoint folds of equal size. Each
  fold uses $K-1$ parts as in-bag (weight 1) and 1 part as out-of-bag
  (weight 0). Every observation is out-of-bag in exactly one fold.

- **Subsampling**: each fold draws a random subset of $\lfloor fn \rfloor$
  observations (default $f = 0.5$). In-bag observations get weight 1;
  out-of-bag observations get weight 0. Observations can be in-bag in
  multiple folds or in none.

```{note}
The penalized learners **recompute** their penalty multiplier $\lambda$
under each fold's weights, because $\mathrm{df}(\lambda)$ depends on $W$. A
fold that down-weights some observations changes the effective sample size
and therefore the $\lambda$ needed to hit a target `df`. This per-fold
re-solve is the dominant cost of `cvrisk` for spline-heavy models — see
[](penalties_df.ipynb).
```

Because the cyclic boosting increments are fixed once fit, the risk at every $m$
is read off a single fitted path per fold — no refitting per grid point. Folds
are embarrassingly parallel: the GIL is released during each fold, and folds run
in parallel via Rayon in the Rust engine.

### The `risk=` parameter

`boost_control(risk=...)` selects which risk is recorded during fitting:

| `risk` | What is recorded | Impact on path |
|---|---|---|
| `"inbag"` (default) | Risk on the training (in-bag) sample | Standard; the path is fitted to this risk |
| `"oobag"` | Risk on the out-of-bag subsample | Requires `weights` that define in-bag vs OOB; used internally by `cvrisk` |
| `"none"` | No risk recorded (faster, no risk array) | For prediction-only fits |

The recorded `risk_` path has length `mstop + 1` (index 0 is the offset-only
risk). For `"inbag"`, this is the in-sample risk; for `"oobag"`, it is the
out-of-sample risk on the held-out subsample.

## Choosing $m_{\text{stop}}$: practical guidance

### Choosing the number of folds

- **Bootstrap**: $B = 25$ is typical; 50 reduces variance further. Each fold is a
  full fit, so cost scales linearly in $B$.
- **K-fold**: $B = 5$ or $10$ is standard (5-fold or 10-fold CV). Larger $B$
  gives more training data per fold but fewer OOB points per fold.
- **Subsampling**: $B = 25$ with `frac` = 0.5 (half in-bag, half OOB). A good
  default when you want deterministic control of the subsample size.

### Flat or monotone risk curves

If the cross-validated risk monotonically decreases up to $m_{\max}$, the model
has not overfit yet — increase `mstop` and re-run. If the curve is nearly flat
across all iterations:

1. **The model is correctly specified but weak.** Try increasing `df` for smooth
   terms (e.g. `df=6` instead of `df=4`) or adding more base learners.
2. **The signal-to-noise ratio is low.** Cross-validated risk near
   the null model (offset only) means there is little to learn from the data.
3. **The family is wrong.** A quantile model at $\tau=0.1$ on symmetric data may
   show a flat risk because the asymmetric loss is dominated by the symmetric
   noise.

### High-variance risk curves

With small datasets or binary outcomes, the folds may disagree substantially.
Remedies:

- Repeat the split or increase $B$ to measure split sensitivity; overlapping
  training sets mean more folds do not guarantee lower variance.
- Use `kfold` instead of `bootstrap` for less optimistic bias.
- Accept the imprecision and pick $m_{\text{stop}}$ conservatively (the "one
  standard error rule": choose the simplest model within 1 SE of the minimum).

## Weights

Observation weights $w_i$ enter the risk $\sum_i w_i\,\rho(y_i, f_i)$ and the
per-observation squared error used for base learner selection. They are useful
for:

- **Case-control oversampling**: assign weight 2 to minority-class observations,
  weight 1 to majority.
- **Frequency weights**: integer weights representing duplicate observations.
- **Variance weights**: if observation $i$ has known variance $\sigma_i^2$, set
  $w_i = 1/\sigma_i^2$.
- **Survey weights**: design weights for complex sampling.

Weights interact with `cvrisk`: each fold reweights observations, and the
out-of-bag risk uses the original weights (not the fold weights). This matches
R `mboost`'s convention.

### Weight dependence of `df`

The `df` to `λ` mapping depends on $W$. When `cvrisk` reweights observations per
fold (zeroing out-of-bag weights), `λ` is recomputed so the effective `df`
stays constant. **This is essential for parity with R**, and it means a term
with `df=4` has exactly 4 effective degrees of freedom in every fold, regardless
of the fold's effective sample size. See [](penalties_df.ipynb) for the mechanics.

## Offsets

The `offset=` argument to `glmboost`/`gamboost` sets the initial predictor
$\hat f^{[0]}$ to a known value instead of the family's default offset. Use it
to:

- **Warm-start** from a previous fit: `offset=fit.fitted_` starts from the
  predictor of an existing model.
- **Fix a known intercept**: `offset=np.log(prevalence)` for Binomial models
  with a known baseline rate.
- **Include an external covariate**: `offset=external_prediction` adds a
  known linear predictor before boosting starts.

If `offset=` is not provided, the family computes it automatically (e.g.,
weighted mean for Gaussian, logit of the event rate for Binomial).

## Information criteria

For families with a likelihood, an AIC-type criterion {cite}`akaike1974new` avoids resampling. The
general form is a penalized log-likelihood:

$$
\mathrm{AIC}(m) = -2\,\ell\big(\hat f^{[m]}\big) + 2\,\mathrm{df}(m),
$$

where $\ell$ is the log-likelihood (evaluated at the fitted predictor) and
$\mathrm{df}(m) = \operatorname{tr}(2 B_m - B_m B_m^\top)$ is the boosting hat
matrix's degrees of freedom. The penalty $2\,\mathrm{df}(m)$ approximates the
expected optimism of the training log-likelihood as an estimate of the test
log-likelihood, under the assumption that the hat-matrix approximation is
accurate.

Both `glmboost` and `gamboost` support the `.aic()` method:

```python
aic = fit.aic()                 # works for both GLMBoostModel and GAMBoostModel
print(f"Optimal mstop: {aic.mstop}")
mb.plot.plot_aic(aic)
fit_truncated = fit[:aic.mstop]
```


For `glmboost`, the hat-matrix trace is computed from the linear (bols)
base-learners; for `gamboost`, it uses the spline and other smooth base-learners'
hat matrices. The AIC is not available for base-learners without hat matrices
(btree, bmono) or for Multinomial models (Kronecker-product learners).

### The corrected Gaussian stopping criterion

`aic(method="corrected")` matches `mboost`'s per-observation Gaussian
stopping criterion {cite}`hurvich1989regression`:

$$
C(m) = \log\left(\frac{\mathrm{RSS}(m)}{n}\right) +
       \frac{1 + \mathrm{df}(m)/n}{1 - (\mathrm{df}(m)+2)/n}.
$$

This value is not the textbook total AICc and is not on a total GAIC scale.
`AicResult` therefore names it `mboost_corrected` with normalization
`per_observation`. Its hat-matrix trace is conditional on the selected path
and does not include the extra degrees of freedom spent searching learners.

### gMDL: generalized minimum description length

For Gaussian families, an alternative criterion is the gMDL {cite}`hansen2001model`
(generalized minimum description length), which does not require an estimate of
the error variance $\sigma^2$. It is derived from a universal prior on the
signal-to-noise ratio and takes the form

$$
s_m = \frac{\mathrm{RSS}(m)}{n-\mathrm{df}(m)}, \qquad
\mathrm{gMDL}(m) = \log(s_m) + \frac{\mathrm{df}(m)}{n}
\log\left(\frac{\sum_i y_i^2-\mathrm{RSS}(m)}{\mathrm{df}(m)s_m}\right).
$$

The response sum of squares is raw and unweighted, matching upstream; residual
risk and $n$ use the fit's effective weights. Pass `method="gmdl"`.

### AIC compared with cross-validation

| Criterion | Pros | Cons |
|---|---|---|
| `cvrisk` | Works for every family; direct estimate of test risk | Computationally expensive ($B$ full refits) |
| AIC | No resampling; fast; deterministic | Requires a likelihood; df approximation may be poor for small $n$; no AIC for tree learners or multinomial Kronecker models |

**Recommendation:** Use `cvrisk` as the default. Use AIC when resampling is
too expensive (large $n$, many base learners), or as a fast
cross-check. If AIC and `cvrisk` disagree substantially, trust `cvrisk`.



## Verifying the criteria: Friedman #1

The claim to check: `cvrisk`, AIC, and the true (oracle) test-risk minimizer
should all land in roughly the same neighborhood of $m_{\text{stop}}$. Use
Friedman's classic nonlinear regression benchmark {cite}`friedman1991multivariate`,
which mixes a few informative predictors with pure-noise ones, and compare
against an oracle that gets to evaluate risk against the *noiseless* signal
on a large held-out set (something `cvrisk` and AIC never have access to).

In [1]:
import warnings
import numpy as np
import mboostrs as mb

warnings.filterwarnings("ignore", message="Some 'x' values are beyond")

# mboostrs.datasets.friedman1 is the library's own Friedman #1 generator
# (Friedman, 1991); it returns the noiseless signal alongside y for
# exactly this kind of oracle comparison.
n_train, n_test = 400, 4000
df_train = mb.datasets.friedman1(n=n_train, n_features=10, noise=1.0, seed=0)
df_test = mb.datasets.friedman1(n=n_test, n_features=10, noise=1.0, seed=1)

cols = [f"x{i}" for i in range(10)]
data_train = {c: df_train[c].to_numpy() for c in cols}
data_train["y"] = df_train["y"].to_numpy()
data_test = {c: df_test[c].to_numpy() for c in cols}
y_test_signal = df_test["y_signal"].to_numpy()

formula = "y ~ " + " + ".join(f"bbs({c}, df=4)" for c in cols)
mstop_max = 400
fit = mb.gamboost(formula, data=data_train, family=mb.Gaussian(),
                   control=mb.boost_control(mstop=mstop_max, nu=0.1))

# Oracle: risk against the noiseless signal on a held-out set, at every m.
grid = np.arange(0, mstop_max + 1, 5)
oracle_risk = np.array([
    np.mean((np.full(n_test, fit.offset_) if m == 0
             else fit[:m].predict(newdata=data_test, type="response") - y_test_signal) ** 2)
    if m == 0 else
    np.mean((fit[:m].predict(newdata=data_test, type="response") - y_test_signal) ** 2)
    for m in grid
])
m_oracle = int(grid[np.argmin(oracle_risk)])

cvr = fit.cvrisk(type="kfold", B=10, seed=0)
aic = fit.aic()

print(f"oracle argmin mstop  = {m_oracle}")
print(f"cvrisk optimal mstop = {cvr.optimal_mstop}")
print(f"AIC optimal mstop    = {aic.mstop}")

# The oracle minimum should be a genuine interior point (risk has turned back
# up by mstop_max), not still descending at the boundary.
assert oracle_risk[-1] > oracle_risk.min()
assert 20 < m_oracle < mstop_max - 20

# cvrisk and AIC should land in the same neighborhood as the oracle, not off
# by a large multiple.
assert abs(cvr.optimal_mstop - m_oracle) < 100
assert abs(aic.mstop - m_oracle) < 100


oracle argmin mstop  = 220
cvrisk optimal mstop = 167
AIC optimal mstop    = 190


On this run, the oracle test-risk minimum is at
$m_{\text{stop}} \approx 220$; `cvrisk` (10-fold) picks $\approx 167$ and
the corrected Gaussian criterion picks $\approx 190$ — both within the same neighborhood as the oracle
and well short of $m_{\max}=400$ (where the oracle risk has already turned
back up). AIC is markedly cheaper (one fit vs. 10 refits) and came closer
to the oracle than `cvrisk` here (30 vs. 53 iterations off) —
consistent with the chapter's advice to prefer `cvrisk` in general and use
AIC as a fast sanity check.

## Post-selection inference

Boosting performs variable selection at every step. The coefficients, p-values,
and confidence intervals produced by `coef()` and `confint()` are *conditional on
the selected model* — they do not account for the selection uncertainty. This
means:

- Standard errors are too small (they condition on the selected set).
- Confidence intervals undercover (they are too narrow).
- P-values are anti-conservative.

Two approaches address this:

1. **Stability selection** ([](stability_selection.ipynb)): provides finite-sample
   error control on the set of selected variables, giving formal guarantees
   about false positives.
2. **Selective inference**: adjusts confidence intervals for the selection event.
   This is not yet implemented in `mboostrs`; use `confint()` as a descriptive
   tool, not for formal inference.

## Distributional and multi-parameter models

For `gamboostLSS`, $m_{\text{stop}}$ is a vector (one per distribution parameter)
under cyclic fitting; `cvrisk_glmboostLSS` searches the joint grid using the
joint out-of-bag negative log-likelihood. Non-cyclic fitting reduces this to a
single $m_{\text{stop}}$. See [](gamlss.ipynb) for details.

## Practical guidance

- Fix $\nu$ small (0.1) and tune only $m_{\text{stop}}$.
- Use enough $m_{\max}$ that the cross-validated risk curve clearly turns up; if
  the minimum is at the boundary, increase $m_{\max}$.
- Choose the fold count for the resampling design and compute budget. More
  folds do not guarantee a lower-variance selected $\hat m$ because their
  training sets overlap more strongly.
- Selection is *post hoc*: inference conditional on $\hat m$ understates
  uncertainty (see [](stability_selection.ipynb) for a procedure with formal error
  control).
- If the risk curve is noisy, increase $B$ or use `kfold`. If it never
  turns up, increase `mstop` or add richer learners.

## Summary

- Cross-validation estimates held-out loss for a specified resampling design.
  The corrected Gaussian and gMDL criteria are cheaper conditional-path
  approximations with different units and assumptions.
- The criteria use a conditional boosting hat-matrix trace. It omits adaptive
  learner-search degrees of freedom, so it is not a general post-selection
  information criterion.
- The three risk-recording options (inbag, oobag, and none) differ in what is
  recorded during fitting. Cross-validation always uses out-of-bag risk,
  independent of the fitting risk.

## See also

- [](functional_gradient_boosting.ipynb) — the $B_m$ hat matrix and df derivation.
- [](stability_selection.ipynb) — variable selection with formal error control
  (complementary to CV-based stopping).
- [](penalties_df.ipynb) — the per-learner df-to-$\lambda$ mapping.
- [](../usage/model_selection) — runnable examples of cvrisk, AIC, and gMDL.